# AI Lecture Tutor — Colab model and local API demo

Run cells in order. Select a GPU runtime in Runtime → Change runtime type if available; GPU access and session length are not guaranteed. A restart clears uploads. This notebook starts FastAPI on localhost for calls made from this notebook only. Do not expose it through a tunnel: Google Colab managed runtimes prohibit unrelated web services. For a hosted frontend-to-backend deployment, configure an independently permitted backend host.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# If the repository is not already available in /content, paste its public Git URL here.
REPO_URL = ""
REPO_DIR = Path('/content/ai-lecture-tutor')
if REPO_URL:
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / 'backend' / 'api.py').exists():
    raise RuntimeError('Set REPO_URL to the public GitHub URL for this project, then rerun this cell.')
os.chdir(REPO_DIR)
print('Project directory:', REPO_DIR)

In [ ]:
%pip install -q -r backend/requirements.txt

In [ ]:
import getpass, os, torch
from backend.llm.runtime import runtime
from backend.config import MODEL_ID

secret = getpass.getpass('Enter a long random BACKEND_SECRET for local API calls: ')
if len(secret) < 24:
    raise ValueError('Use at least 24 characters for the shared secret.')
os.environ['BACKEND_SECRET'] = secret
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Model loading/generation may be very slow and may exceed available RAM.')
print('Loading', MODEL_ID, 'and sentence-transformers/all-MiniLM-L6-v2. First load downloads several GB of model data.')
runtime.load()
print('Model and local embedding model ready on', runtime.device)

In [ ]:
import subprocess, sys, time, urllib.request

api = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'backend.api:app', '--host', '127.0.0.1', '--port', '8000'], cwd=str(REPO_DIR))
for _ in range(60):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as response:
            print('Backend health:', response.read().decode())
            break
    except Exception:
        if api.poll() is not None:
            raise RuntimeError('FastAPI stopped during startup; inspect the cell output.')
        time.sleep(1)
else:
    raise TimeoutError('FastAPI did not become ready within 60 seconds.')

In [ ]:
print('Local API URL (not public): http://127.0.0.1:8000')
print('Use this URL for authenticated requests from this notebook runtime only.')